# 📷 Photomosaic 生成器

本地 Mac 玩具。把 `pool/` 里的一堆照片重组成 `target.jpg`。

**用法**: Run All → 看 Cell 4 里的滑条 → 点预览 / 正式渲染。

In [ ]:
from pathlib import Path
from IPython.display import display, Image as IPImage, Markdown
import ipywidgets as widgets

from mosaic.config import DEFAULT_CONFIG, build_widgets
from mosaic.pipeline import run_pipeline

In [ ]:
CFG = dict(DEFAULT_CONFIG)
# 改下面三项指向你的实际路径
CFG['target_path'] = 'target.jpg'
CFG['pool_dir'] = 'pool/'
CFG['cache_dir'] = '.cache'
CFG['output_dir'] = 'output'

Path(CFG['cache_dir']).mkdir(exist_ok=True)
Path(CFG['output_dir']).mkdir(exist_ok=True)
print('config:', CFG)

In [ ]:
ui = build_widgets()
output = widgets.Output()

def _run(is_preview):
    params = ui['get_params']()
    with output:
        output.clear_output()
        grid_cols = CFG['preview_grid_cols'] if is_preview else CFG['grid_cols']
        grid_rows = CFG['preview_grid_rows'] if is_preview else CFG['grid_rows']
        tile_px = CFG['preview_tile_px'] if is_preview else CFG['tile_px']
        print(f"{'预览' if is_preview else '正式'}: {grid_cols}x{grid_rows} @ {tile_px}px")
        result = run_pipeline(
            target_path=CFG['target_path'], pool_dir=CFG['pool_dir'],
            grid_cols=grid_cols, grid_rows=grid_rows, tile_px=tile_px,
            lambda_reuse=params['lambda_reuse'], mu_neighbor=params['mu_neighbor'],
            tau_transfer=params['tau_transfer'],
            cache_path=Path(CFG['cache_dir']) / 'pool_features.pkl',
            output_dir=CFG['output_dir'],
            topk_candidates=CFG['topk_candidates'],
            neighbor_sigma=CFG['neighbor_sigma'],
            do_deepzoom=not is_preview,
            do_report=not is_preview,
        )
        display(IPImage(str(result['png_path'])))
        if not is_preview:
            display(Markdown(result['report_path'].read_text()))
            print(f"DeepZoom: {result['deepzoom_html']}")

ui['preview_button'].on_click(lambda _: _run(is_preview=True))
ui['render_button'].on_click(lambda _: _run(is_preview=False))
display(ui['container'], output)

## 使用提示

- `λ` 大 → tile 不重复；`λ=0` → 谁近用谁
- `μ` 大 → 相邻 tile 会被逼得不一样
- `τ=0` 近看照片清晰，`τ=1` 远看完美。甜区 `[0.4, 0.6]`
- 正式渲染会生成 PNG + Markdown 报告 + DeepZoom HTML，在 `output/`